# <p style="color:#07d839; font-family:sans-serif;"> 😀 Sentiment Analysis — Análisis de Sentimiento 🥲 </p>

---

## <p style="color:#07d839;">🎯 Objetivo de la Tarea</p>

El propósito de este análisis es desentrañar la carga emocional oculta tras las letras de diferentes géneros musicales. Buscamos responder preguntas clave sobre la naturaleza sentimental de la música:

*   **Polaridad:** ¿Qué géneros son predominantemente positivos, negativos o neutros?
*   **Intensidad:** ¿Existen géneros con una carga emocional negativa o positiva mucho más marcada?
*   **Emociones específicas:** ¿Qué géneros destacan en sentimientos como la **tristeza**, **ira**, **alegría** o **amor**?
*   **Variabilidad:** ¿Es el sentimiento constante dentro de un género o existe mucha disparidad entre canciones?

---

## <p style="color:#07d839;">⚙️ Metodología del Análisis</p>

La unidad básica de estudio es **cada canción individual**. El flujo de procesamiento sigue esta estructura lógica:

> **Flujo de Trabajo:**
> `Canción` ➔ `Letra (Texto)` ➔ `Modelo de Sentimiento` ➔ `Score (Positivo/Negativo/Neutro)`

### Procesamiento por Género
Una vez obtenido el score individual, agruparemos los resultados para obtener una visión panorámica por categoría:

| Género Musical | Métricas a Calcular |
| :--- | :--- |
| **Pop** | Media de sentimiento, % Positivas, % Negativas, % Neutras |
| **Rap** | Media de sentimiento, % Positivas, % Negativas, % Neutras |
| **Rock** | Media de sentimiento, % Positivas, % Negativas, % Neutras |
| **...** | ... |



---

In [428]:
# librerías
import pandas as pd
import glob
import os
import numpy as np
import re
import ast
import html
from pathlib import Path
import torch
import gc
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import matplotlib.pyplot as plt


---
### 1. Limpieza dataset

+ Eliminar canciones sin lyrics.

+ Eliminar canciones sin ningún género válido.

+ Quitar letras demasiado cortas, por ejemplo menos de 30 o 50 palabras.

+ Normalizar nombres de géneros: pop, Pop, POP → pop.

+ Revisar duplicados por artist + song.

+ NLP sobre lyrics para preparar análisis de sentimiento 

Limpiar la letra sin destruir información útil para el sentimiento. En análisis de sentimiento no conviene eliminar stopwords agresivamente, porque palabras como not, never, no, without cambian completamente el significado.

In [429]:
# Fusionar Datasets - !!!!! SOLO EJECUTAR UNA VEZ !!!!!!

patron_busqueda = "music_dataset_*.csv"
archivos_csv = glob.glob(patron_busqueda)

# Verificamos
if not archivos_csv:
    print("No se encontraron archivos con el prefijo 'music_dataset_'. Revisa el directorio.")
else:
    print(f"Archivos encontrados para fusionar: {archivos_csv}")
    
    # Leer cada archivo CSV y almacenarlo en una lista
    lista_dataframes = []
    
    for archivo in archivos_csv:
        # Leemos el CSV
        df = pd.read_csv(archivo)
        lista_dataframes.append(df)
        print(f"Leído: {archivo} con {len(df)} filas.")

    # Concatenar (fusionar) todos los DataFrames de la lista en uno solo
    # ignore_index=True es importante para que el índice se reinicie (0, 1, 2...) 
    # en lugar de mantener los índices originales de cada archivo por separado.
    df_final = pd.concat(lista_dataframes, ignore_index=True)

    # Guardar el resultado en un nuevo archivo CSV
    nombre_archivo_salida = "music_dataset_final.csv"
    
    # index=False evita que se guarde la columna de números de fila en el CSV final
    df_final.to_csv(nombre_archivo_salida, index=False)

    print(f"\n¡Fusión completada! El archivo final tiene {len(df_final)} filas en total.")
    print(f"Se ha guardado como: {nombre_archivo_salida}")

Archivos encontrados para fusionar: ['music_dataset_20000_30000.csv', 'music_dataset_40000_50000.csv', 'music_dataset_30000_40000.csv', 'music_dataset_50000_70000.csv', 'music_dataset_jose.csv']
Leído: music_dataset_20000_30000.csv con 1035 filas.
Leído: music_dataset_40000_50000.csv con 101 filas.
Leído: music_dataset_30000_40000.csv con 801 filas.
Leído: music_dataset_50000_70000.csv con 602 filas.
Leído: music_dataset_jose.csv con 201 filas.

¡Fusión completada! El archivo final tiene 2740 filas en total.
Se ha guardado como: music_dataset_final.csv


In [430]:
# cargar dataset
DATA_PATH = "music_dataset_final.csv"

df = pd.read_csv(DATA_PATH)

print("Shape inicial:", df.shape)
display(df.head())
display(df.tail())

Shape inicial: (2740, 7)


,artist,song,lyrics,genres,genre_1,genre_2,genre_3
0,Tarja Turunen,Until Silence,NaN,"symphonic metal, female vocalists, female fron...",symphonic metal,female vocalists,female fronted metal
1,Susan Boyle,You Have To Be There,What is it Lord that you want\n\nThat I am not...,"pop, love it, female vocalists",pop,love it,female vocalists
2,Taylor Swift,Cruel Summer,"(Yeah, yeah, yeah, yeah)\n\n\n\nFever dream hi...","electropop, pop, synthpop",electropop,pop,synthpop
3,Taylor Swift,London Boy,[Idris Elba and James Corden:]\n\nWe can go dr...,NaN,NaN,NaN,NaN
4,Taylor Swift,Sugar,What a thing to see\n\nWhat a thing to be\n\nW...,NaN,NaN,NaN,NaN


,artist,song,lyrics,genres,genre_1,genre_2,genre_3
2735,Sarah Brightman,The Music of the Night,"Nighttime sharpens, heightens each sensation.\...","classical, female vocalists, opera",classical,female vocalists,opera
2736,The Phantom Of The Opera,Wandering Child,NaN,NaN,NaN,NaN,NaN
2737,Placido Domingo,Yesterday,NaN,"opera, classical, tenor",opera,classical,tenor
2738,Tarja Turunen,Goldfinger,NaN,"symphonic metal, female vocalists, finnish",symphonic metal,female vocalists,finnish
2739,Tarja Turunen,Until Silence,NaN,"symphonic metal, rock, tarja turunen",symphonic metal,rock,tarja turunen


In [431]:
# Normalizar columnas
df.columns = df.columns.str.strip().str.lower()

expected_cols = ["artist", "song", "lyrics", "genres", "genre_1", "genre_2", "genre_3"]

missing_cols = [col for col in expected_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas en el dataset: {missing_cols}")

print("Columnas disponibles:")
print(df.columns.tolist())

Columnas disponibles:
['artist', 'song', 'lyrics', 'genres', 'genre_1', 'genre_2', 'genre_3']


### 1.1 NLP

**PREPROCESADO DE LYRICS**

In [432]:
def clean_lyrics(text):
    """
    Limpieza conservadora de letras para análisis emocional.
    No elimina stopwords ni aplica stemming/lemmatization.
    """
    
    if pd.isna(text):
        return np.nan
    
    text = str(text)
    
    # Decodificar entidades HTML
    text = html.unescape(text)
    
    # Eliminar URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    
    # Normalizar comillas raras
    text = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("‘", "'")
            .replace("’", "'")
    )
    
    # Eliminar etiquetas típicas de letras
    section_patterns = [
        r"\[.*?(verse|chorus|bridge|intro|outro|hook|pre-chorus|post-chorus|interlude|refrain).*?\]",
        r"\(.*?(verse|chorus|bridge|intro|outro|hook|pre-chorus|post-chorus|interlude|refrain).*?\)"
    ]
    
    for pattern in section_patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    
    # Eliminar patrones frecuentes de Genius u otras fuentes
    text = re.sub(r"\d+\s*contributors?", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"you might also like", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"embed$", " ", text, flags=re.IGNORECASE)
    
    # Normalizar saltos de línea y tabs
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")
    
    # Espacios múltiples
    text = re.sub(r"\s+", " ", text).strip()
    
    if text == "":
        return np.nan
    
    return text

df["lyrics_clean"] = df["lyrics"].apply(clean_lyrics)

def count_words(text):
    if pd.isna(text):
        return 0
    return len(re.findall(r"\b[\w']+\b", str(text)))

df["word_count"] = df["lyrics_clean"].apply(count_words)

display(df[["artist", "song", "lyrics_clean", "word_count"]].head())

,artist,song,lyrics_clean,word_count
0,Tarja Turunen,Until Silence,NaN,0
1,Susan Boyle,You Have To Be There,What is it Lord that you want That I am not se...,325
2,Taylor Swift,Cruel Summer,"(Yeah, yeah, yeah, yeah) Fever dream high in t...",447
3,Taylor Swift,London Boy,[Idris Elba and James Corden:] We can go drivi...,443
4,Taylor Swift,Sugar,What a thing to see What a thing to be What a ...,241


- No hemos eliminado stopwords.
- No hemos hecho stemming.
- No hemos hecho lemmatization.
- No hemos eliminado signos de puntuación de forma agresiva.
- No hemos convertido todo a minúsculas.
- No hemos eliminado negaciones como "not", "no", "never".
- No hemos tokenizado manualmente para entrenar un modelo clásico.

En la fase de preprocesado de letras se aplicó una limpieza conservadora orientada al uso posterior de un modelo Transformer de clasificación emocional. Se eliminaron elementos no pertenecientes a la letra, como URLs, etiquetas estructurales de canciones ([Verse], [Chorus], [Bridge]), patrones procedentes del scraping y espacios duplicados. También se normalizaron comillas y entidades HTML para mejorar la consistencia del texto.

No se aplicaron técnicas agresivas como eliminación de stopwords, stemming o lematización, ya que el modelo emocional necesita conservar el contexto lingüístico completo de las frases. Finalmente, se calculó el número de palabras de cada letra y se descartaron canciones sin letra válida o con letras demasiado cortas.

**PREPROCESADO DE GENRES**

In [433]:
VALID_GENRES = {
    "alt country",
    "alt rock",
    "alt-country",
    "alternative",
    "alternative metal",
    "alternative rock",
    "ambient",
    "american country",
    "art rock",
    "black metal",
    "bluegrass",
    "blues",
    "blues rock",
    "blues-rock",
    "bossa nova",
    "canadian country",
    "celtic",
    "chicago blues",
    "chillout",
    "classic blues",
    "classic country",
    "classic rock",
    "classical",
    "contemporary country",
    "contry",
    "country",
    "country and western",
    "country pop",
    "country rap",
    "country rock",
    "country two step",
    "country-pop",
    "dance",
    "deep house",
    "delta blues",
    "disco",
    "dream pop",
    "electro house",
    "electro pop",
    "electroclash",
    "electronic",
    "electropop",
    "emo",
    "emocore",
    "experimental",
    "folk",
    "folk pop",
    "folk rock",
    "folkrock",
    "funk",
    "future bass",
    "gospel",
    "gothic metal",
    "hard rock",
    "hip hop",
    "honky tonk",
    "house",
    "indie",
    "indie folk",
    "indie pop",
    "indie rock",
    "indietronica",
    "jazz",
    "melodic death metal",
    "metal",
    "modern country",
    "neo-soul",
    "new age",
    "new traditionalist",
    "opera",
    "outlaw country",
    "pop",
    "pop country",
    "pop punk",
    "pop rock",
    "pop soul",
    "post-hardcore",
    "progressive folk rock",
    "progressive rock",
    "psychedelic",
    "psychedelic rock",
    "r&b",
    "rap",
    "rock",
    "rock n roll",
    "rockabilly",
    "screamo",
    "soft rock",
    "soul",
    "southern rock",
    "symphonic metal",
    "synthpop",
    "trap",
    "trip-hop"
}

GENRE_MAPPING = {
    # Hip hop
    "hip-hop": "hip hop",
    "hiphop": "hip hop",

    # R&B
    "rnb": "r&b",
    "r and b": "r&b",
    "rhythm and blues": "r&b",

    # Rock
    "rock and roll": "rock",
    "rock & roll": "rock",
    "rock n' roll": "rock n roll",

    # Electronic
    "edm": "electronic",
    "electronic dance music": "electronic",

    # Country
    "contry": "country",
    "country-pop": "country pop",
    "alt-country": "alt country",

    # Folk / rock variants
    "folkrock": "folk rock",
    "blues-rock": "blues rock",

    # Singer-songwriter
    "singer-songwriter": "singer songwriter",

    # Soul variants
    "neo soul": "neo-soul",

    # Trip hop
    "trip hop": "trip-hop"
}


def normalize_genre(genre):
    """
    Normaliza una etiqueta de género individual.
    Solo devuelve el género si pertenece a VALID_GENRES.
    Si no pertenece, devuelve None.
    """

    if pd.isna(genre):
        return None

    genre = str(genre)
    genre = html.unescape(genre)
    genre = genre.strip().lower()

    # Quitar comillas externas
    genre = genre.strip("'").strip('"')

    # Normalizar separadores simples
    genre = genre.replace("_", " ")
    genre = re.sub(r"\s+", " ", genre).strip()

    # Quitar puntuación rara al principio/final, pero mantener símbolos internos como r&b o trip-hop
    genre = re.sub(r"^[^\w]+|[^\w]+$", "", genre)

    # Aplicar mapeo de variantes a una forma más estándar
    genre = GENRE_MAPPING.get(genre, genre)

    # Aceptar solo géneros válidos
    if genre not in VALID_GENRES:
        return None

    return genre


def parse_genre_cell(value):
    """
    Convierte una celda de género en una lista de géneros normalizados.
    Sirve para celdas simples, listas en string o valores separados por comas.
    Solo conserva géneros incluidos en VALID_GENRES.
    """

    if pd.isna(value):
        return []

    value = str(value).strip()

    parsed_values = []

    # Caso: "['pop', 'rock']"
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, (list, tuple, set)):
                parsed_values = list(parsed)
            else:
                parsed_values = [value]
        except Exception:
            parsed_values = [value]

    # Caso: "pop, rock, r&b"
    elif "," in value:
        parsed_values = value.split(",")

    # Caso simple: "pop"
    else:
        parsed_values = [value]

    normalized = []

    for genre in parsed_values:
        genre_norm = normalize_genre(genre)
        if genre_norm is not None:
            normalized.append(genre_norm)

    # Eliminar duplicados manteniendo el orden
    unique_normalized = []
    seen = set()

    for genre in normalized:
        if genre not in seen:
            unique_normalized.append(genre)
            seen.add(genre)

    return unique_normalized

In [434]:
# NORMALIZAR GENERO

genre_cols = ["genre_1", "genre_2", "genre_3"]

for col in genre_cols:
    df[col + "_clean"] = df[col].apply(
        lambda x: parse_genre_cell(x)[0] if len(parse_genre_cell(x)) > 0 else None
    )

display(df[["genres", "genre_1", "genre_2", "genre_3", 
            "genre_1_clean", "genre_2_clean", "genre_3_clean"]].head())

,genres,genre_1,genre_2,genre_3,genre_1_clean,genre_2_clean,genre_3_clean
0,"symphonic metal, female vocalists, female fron...",symphonic metal,female vocalists,female fronted metal,symphonic metal,NaN,NaN
1,"pop, love it, female vocalists",pop,love it,female vocalists,pop,NaN,NaN
2,"electropop, pop, synthpop",electropop,pop,synthpop,electropop,pop,synthpop
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**GENEROS POR CANCIÓN**

In [435]:
def build_genre_list(row):
    """
    Construye una lista de géneros únicos a partir de genre_1_clean,
    genre_2_clean y genre_3_clean.
    """
    
    genres = []
    
    for col in ["genre_1_clean", "genre_2_clean", "genre_3_clean"]:
        genre = row[col]
        if genre is not None and pd.notna(genre):
            genres.append(genre)
    
    # Eliminar duplicados manteniendo el orden
    unique_genres = []
    seen = set()
    
    for genre in genres:
        if genre not in seen:
            unique_genres.append(genre)
            seen.add(genre)
    
    return unique_genres


df["genre_list"] = df.apply(build_genre_list, axis=1)
df["n_genres"] = df["genre_list"].apply(len)

display(df[["artist", "song", "genre_list", "n_genres"]].head())

,artist,song,genre_list,n_genres
0,Tarja Turunen,Until Silence,[symphonic metal],1
1,Susan Boyle,You Have To Be There,[pop],1
2,Taylor Swift,Cruel Summer,"[electropop, pop, synthpop]",3
3,Taylor Swift,London Boy,[],0
4,Taylor Swift,Sugar,[],0


In [436]:
MIN_WORDS = 1

initial_rows = len(df)

mask_valid_lyrics = df["lyrics_clean"].notna() & (df["word_count"] >= MIN_WORDS)
mask_valid_genres = df["n_genres"] > 0

df_clean = df[mask_valid_lyrics & mask_valid_genres].copy()

print("Filas iniciales:", initial_rows)
print("Filas tras limpieza:", len(df_clean))
print("Canciones eliminadas:", initial_rows - len(df_clean))

print("\nMotivos principales:")
print("Sin letra válida o letra demasiado corta:", (~mask_valid_lyrics).sum())
print("Sin género válido:", (~mask_valid_genres).sum())

Filas iniciales: 2740
Filas tras limpieza: 1380
Canciones eliminadas: 1360

Motivos principales:
Sin letra válida o letra demasiado corta: 1063
Sin género válido: 619


In [437]:
def normalize_text_id(text):
    """
    Normaliza texto para detectar duplicados.
    No se usa para el modelo, solo para comparar artist/song.
    """
    
    if pd.isna(text):
        return ""
    
    text = str(text).strip().lower()
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text)
    
    return text


df_clean["artist_norm"] = df_clean["artist"].apply(normalize_text_id)
df_clean["song_norm"] = df_clean["song"].apply(normalize_text_id)

duplicates = df_clean.duplicated(subset=["artist_norm", "song_norm"], keep=False)

print("Posibles duplicados:", duplicates.sum())

df_clean = (
    df_clean
    .sort_values("word_count", ascending=False)
    .drop_duplicates(subset=["artist_norm", "song_norm"], keep="first")
    .sort_index()
    .copy()
)

print("Shape tras eliminar duplicados:", df_clean.shape)

Posibles duplicados: 4
Shape tras eliminar duplicados: (1378, 16)


In [438]:
# Dataset expandido por genero

df_genres = df_clean.explode("genre_list").copy()

df_genres = df_genres.rename(columns={"genre_list": "genre"})

df_genres = df_genres[df_genres["genre"].notna()].copy()

# Evitar que una misma canción aparezca dos veces en el mismo género
df_genres = df_genres.drop_duplicates(
    subset=["artist_norm", "song_norm", "genre"],
    keep="first"
)

print("Shape df_clean, una fila por canción:", df_clean.shape)
print("Shape df_genres, una fila por canción-género:", df_genres.shape)

display(df_genres[["artist", "song", "genre", "lyrics_clean", "word_count"]].head())

Shape df_clean, una fila por canción: (1378, 16)
Shape df_genres, una fila por canción-género: (2574, 16)


,artist,song,genre,lyrics_clean,word_count
1,Susan Boyle,You Have To Be There,pop,What is it Lord that you want That I am not se...,325
2,Taylor Swift,Cruel Summer,electropop,"(Yeah, yeah, yeah, yeah) Fever dream high in t...",447
2,Taylor Swift,Cruel Summer,pop,"(Yeah, yeah, yeah, yeah) Fever dream high in t...",447
2,Taylor Swift,Cruel Summer,synthpop,"(Yeah, yeah, yeah, yeah) Fever dream high in t...",447
5,Shania Twain,Forever And For Always,country,"Mm, mm Mm, in your arms Oh I can hear your hea...",389


In [439]:
genre_counts = df_genres["genre"].value_counts()


# 1. Le decimos a Pandas que no ponga límite máximo de filas
#pd.set_option('display.max_rows', None)

# 2. Mostramos tu tabla (sin el .head())
display(
    genre_counts
    .reset_index()
    .rename(columns={"index": "genre", "genre": "n_songs"})
)

# 3. Restauramos el límite normal (por defecto suele ser 60)
# Esto es importante para que futuras tablas grandes no te bloqueen la pantalla
pd.reset_option('display.max_rows')

print("Número total de géneros únicos:", df_genres["genre"].nunique())
print("Número total de registros canción-género:", len(df_genres))


,n_songs,count
0,country,499
1,pop,370
2,rock,209
3,folk,131
4,dance,120
...,...,...
84,experimental,1
85,progressive folk rock,1
86,art rock,1
87,pop soul,1


Número total de géneros únicos: 89
Número total de registros canción-género: 2574


In [440]:
MIN_SONGS_PER_GENRE = 1

valid_genres = genre_counts[genre_counts >= MIN_SONGS_PER_GENRE].index

df_genres_filtered = df_genres[df_genres["genre"].isin(valid_genres)].copy()

print("Géneros antes del filtro:", df_genres["genre"].nunique())
print("Géneros después del filtro:", df_genres_filtered["genre"].nunique())

print("Registros canción-género antes:", len(df_genres))
print("Registros canción-género después:", len(df_genres_filtered))

display(
    df_genres_filtered["genre"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "genre", "genre": "n_songs"})
)

Géneros antes del filtro: 89
Géneros después del filtro: 89
Registros canción-género antes: 2574
Registros canción-género después: 2574


,n_songs,count
0,country,499
1,pop,370
2,rock,209
3,folk,131
4,dance,120
...,...,...
84,experimental,1
85,progressive folk rock,1
86,art rock,1
87,pop soul,1


In [441]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df_clean.to_csv(OUTPUT_DIR / "songs_clean_by_song.csv", index=False)
df_genres.to_csv(OUTPUT_DIR / "songs_clean_by_genre_exploded.csv", index=False)
df_genres_filtered.to_csv(OUTPUT_DIR / "songs_clean_by_genre_exploded_filtered.csv", index=False)

print("Archivos guardados:")
print(OUTPUT_DIR / "songs_clean_by_song.csv")
print(OUTPUT_DIR / "songs_clean_by_genre_exploded.csv")
print(OUTPUT_DIR / "songs_clean_by_genre_exploded_filtered.csv")

Archivos guardados:
outputs/songs_clean_by_song.csv
outputs/songs_clean_by_genre_exploded.csv
outputs/songs_clean_by_genre_exploded_filtered.csv


En primer lugar, se realizó una limpieza del dataset original con el objetivo de preparar las letras y los géneros musicales para el análisis emocional posterior. Se eliminaron canciones sin letra disponible, canciones con letras demasiado cortas y registros sin ningún género válido. La limpieza de las letras fue conservadora, evitando eliminar stopwords o modificar excesivamente el texto, ya que el modelo de emociones utilizado posteriormente necesita conservar el contexto lingüístico de las frases.

Además, se normalizaron las etiquetas de género presentes en las columnas genre_1, genre_2 y genre_3, convirtiéndolas a minúsculas, eliminando espacios innecesarios y unificando algunas variantes frecuentes. Dado que una canción puede pertenecer a varios géneros, se generó un segundo dataset en formato expandido, donde cada fila representa una combinación canción-género. Este formato permite analizar posteriormente la distribución emocional de las letras para cada género musical.

| Archivo                                      | Nivel                       | Uso principal                   |
| -------------------------------------------- | --------------------------- | ------------------------------- |
| `songs_clean_by_song.csv`                    | Una fila por canción        | Aplicar el modelo emocional     |
| `songs_clean_by_genre_exploded.csv`          | Una fila por canción-género | Análisis completo por género    |
| `songs_clean_by_genre_exploded_filtered.csv` | Canción-género, filtrado    | Gráficos y conclusiones fiables |


--- 

### 2. Usar genre_1 como análisis principal y, si hay tiempo, hacer una versión secundaria usando todos los géneros con formato “multi-label”.

---
### 3. Aplicar análisis de sentimiento a cada letra

Para cada canción se calcularía un sentimiento. El resultado ideal sería añadir nuevas columnas al dataset:

+ sentiment_score

+ sentiment_label

+ positive_score

+ neutral_score

+ negative_score

+ word_count


---
### 4.  Clasificación de canciones en sentimiento.

Buscar modelo potente:

+ Opción A: modelo de emociones con 7 clases --> Un modelo como j-hartmann/emotion-english-distilroberta-base clasifica texto en emociones básicas: anger, disgust, fear, joy, neutral, sadness y surprise.

+ Opción B: modelo GoEmotions con 28 emociones --> Un modelo como SamLowe/roberta-base-go_emotions está entrenado sobre el dataset GoEmotions y funciona como clasificación multi-label, es decir, una misma letra puede activar varias emociones a la vez.

1. Cargar songs_clean_by_song.csv
2. Aplicar modelo de 7 emociones a lyrics_clean
3. Crear columnas con scores emocionales
4. Crear columna con emoción dominante
5. Guardar songs_with_emotions.csv
6. Unir esos resultados con songs_clean_by_genre_exploded_filtered.csv
7. Analizar emociones por género

In [442]:
#!pip install transformers torch tqdm -q

In [443]:
# cargar datoset limpio por canción

df_songs = df_clean.copy()

print("Shape:", df_songs.shape)
display(df_songs[["artist", "song", "lyrics_clean", "word_count", "genre_list"]].head())

df_songs = df_songs[df_songs["lyrics_clean"].notna()].copy()
df_songs = df_songs.reset_index(drop=True)

print("Canciones válidas para análisis emocional:", len(df_songs))

Shape: (1378, 16)


,artist,song,lyrics_clean,word_count,genre_list
1,Susan Boyle,You Have To Be There,What is it Lord that you want That I am not se...,325,[pop]
2,Taylor Swift,Cruel Summer,"(Yeah, yeah, yeah, yeah) Fever dream high in t...",447,"[electropop, pop, synthpop]"
5,Shania Twain,Forever And For Always,"Mm, mm Mm, in your arms Oh I can hear your hea...",389,"[country, pop]"
9,Johnny Cash,I Heard The Bells On Christmas Day,I heard the bells on Christmas Day Their old f...,169,"[country, folk]"
11,Johnny Cash,Southwind,Southwind You picked her up in Jacksonville An...,139,"[country, folk]"


Canciones válidas para análisis emocional: 1378


In [444]:
# CONFIGURACIÓN GENERAL

TEXT_COL = "lyrics_clean"

MODEL_CONFIGS = [
    {
        "model_key": "cardiff",
        "model_name": "cardiffnlp/twitter-roberta-base-sentiment-latest",
        "task_type": "single_label",
        "output_col": "cardiff_sentiment",
        "score_col": "cardiff_sentiment_score",
        "top_k": 1,
        "exclude_neutral": False
    },
    {
        "model_key": "jhartmann",
        "model_name": "j-hartmann/emotion-english-distilroberta-base",
        "task_type": "single_label",
        "output_col": "jhartmann_emotion",
        "score_col": "jhartmann_emotion_score",
        "top_k": 1,
        "exclude_neutral": False
    },
    {
        "model_key": "goemotions",
        "model_name": "SamLowe/roberta-base-go_emotions",
        "task_type": "multi_label",
        "output_col": "goemotions_top3",
        "score_col": "goemotions_top3_scores",
        "top_k": 3,
        # Recomendado tras el problema de exceso de neutral.
        # Si quieres permitir neutral en el top 3, cambia a False.
        "exclude_neutral": True
    }
]

In [445]:
# DISPOSITIVO
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")


device = get_device()


In [446]:
# FUNCIONES GENERALES - DIVIDIR LETRAS LARGAS EN CHUNKS
MAX_TOKENS = 500
STRIDE = 50


def split_text_into_chunks(text, tokenizer, max_tokens=MAX_TOKENS, stride=STRIDE):
    """
    Divide una letra larga en fragmentos de tokens.
    Se usa max_tokens=500 para dejar margen a los tokens especiales del modelo.
    """

    if pd.isna(text) or str(text).strip() == "":
        return []

    text = str(text)

    tokenized = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False
    )

    token_ids = tokenized["input_ids"]

    if len(token_ids) == 0:
        return []

    chunks = []
    step = max_tokens - stride

    for start in range(0, len(token_ids), step):
        end = start + max_tokens
        chunk_ids = token_ids[start:end]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        if chunk_text.strip():
            chunks.append(chunk_text.strip())

        if end >= len(token_ids):
            break

    return chunks



# NORMALIZAR ETIQUETAS DE LOS MODELOS

def normalize_label(label, model_key=None):
    """
    Normaliza etiquetas devueltas por los modelos.
    """

    label = str(label).lower().strip()

    # Por seguridad, si algún modelo usa LABEL_0 / LABEL_1
    if model_key == "siebert":
        label_mapping = {
            "label_0": "negative",
            "label_1": "positive",
            "0": "negative",
            "1": "positive"
        }
        label = label_mapping.get(label, label)

    return label



# OBTENER LOS SCORES DE UNA CANCIÓN


def predict_text_scores(
    text,
    tokenizer,
    model,
    labels,
    device,
    task_type,
    batch_size=8
):
    """
    Aplica un modelo a una canción completa.
    1. Divide la letra en chunks.
    2. Predice scores por chunk.
    3. Promedia los scores para obtener una predicción global por canción.
    """

    chunks = split_text_into_chunks(text, tokenizer)

    if len(chunks) == 0:
        return {
            "scores": {label: np.nan for label in labels},
            "n_chunks": 0
        }

    all_scores = []

    for i in range(0, len(chunks), batch_size):
        batch_chunks = chunks[i:i + batch_size]

        inputs = tokenizer(
            batch_chunks,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

            if task_type == "single_label":
                probs = torch.softmax(outputs.logits, dim=1)

            elif task_type == "multi_label":
                probs = torch.sigmoid(outputs.logits)

            else:
                raise ValueError(f"task_type no reconocido: {task_type}")

        all_scores.append(probs.detach().cpu().numpy())

    all_scores = np.vstack(all_scores)
    mean_scores = all_scores.mean(axis=0)

    score_dict = {
        label: float(score)
        for label, score in zip(labels, mean_scores)
    }

    return {
        "scores": score_dict,
        "n_chunks": len(chunks)
    }




# EXTRAER LA PREDICCIÓN FINAL


def extract_prediction_from_scores(
    scores,
    model_key,
    top_k=1,
    exclude_neutral=False
):
    """
    Extrae la predicción final a partir de los scores medios.
    
    Para SieBERT y j-hartmann:
    - devuelve top 1.
    
    Para GoEmotions:
    - devuelve top K emociones.
    """

    valid_scores = {
        emotion: score
        for emotion, score in scores.items()
        if not pd.isna(score)
    }

    if exclude_neutral:
        valid_scores = {
            emotion: score
            for emotion, score in valid_scores.items()
            if emotion != "neutral"
        }

    if len(valid_scores) == 0:
        if top_k == 1:
            return None, np.nan
        else:
            return "", ""

    sorted_scores = sorted(
        valid_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    if top_k == 1:
        top_label, top_score = sorted_scores[0]
        return top_label, top_score

    top_items = sorted_scores[:top_k]

    top_labels = [label for label, score in top_items]
    top_scores = [round(score, 4) for label, score in top_items]

    return ", ".join(top_labels), ", ".join(map(str, top_scores))





In [447]:
# FUNCIÓN PARA EJECUTAR CUALQUIER MODELO - POSTERIORMENTE HAREMOS QUE SE APLIQUE A LOS 3 MODELOS QUE QUEREMOS COMPARAR


def run_model_on_df(
    df,
    model_config,
    text_col=TEXT_COL,
    device=device,
    batch_size=8,
    save_all_scores=False
):
    """
    Aplica un modelo Transformer a todas las canciones de df.
    
    Devuelve un DataFrame con:
    - predicción principal
    - score principal
    - número de chunks
    - opcionalmente, scores de todas las etiquetas
    """

    model_key = model_config["model_key"]
    model_name = model_config["model_name"]
    task_type = model_config["task_type"]
    output_col = model_config["output_col"]
    score_col = model_config["score_col"]
    top_k = model_config["top_k"]
    exclude_neutral = model_config["exclude_neutral"]

    print("=" * 90)
    print(f"Cargando modelo: {model_name}")
    print("=" * 90)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    model.to(device)
    model.eval()

    id2label = model.config.id2label

    labels = [
        normalize_label(id2label[i], model_key=model_key)
        for i in range(len(id2label))
    ]

    print("Etiquetas del modelo:")
    print(labels)

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Aplicando {model_key}"):

        text = row[text_col]

        prediction_result = predict_text_scores(
            text=text,
            tokenizer=tokenizer,
            model=model,
            labels=labels,
            device=device,
            task_type=task_type,
            batch_size=batch_size
        )

        scores = prediction_result["scores"]
        n_chunks = prediction_result["n_chunks"]

        pred_label, pred_score = extract_prediction_from_scores(
            scores=scores,
            model_key=model_key,
            top_k=top_k,
            exclude_neutral=exclude_neutral
        )

        result = {
            output_col: pred_label,
            score_col: pred_score,
            f"{model_key}_n_chunks": n_chunks
        }

        if save_all_scores:
            for label, score in scores.items():
                result[f"{model_key}_{label}"] = score

        results.append(result)

    df_results = pd.DataFrame(results)

    # Liberar memoria
    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return df_results

In [448]:
# APLICAR A LOS 3 MODELOS

df_model_comparison = df_songs.copy().reset_index(drop=True)

# Aseguramos que solo analizamos canciones con letra limpia válida
df_model_comparison = df_model_comparison[
    df_model_comparison[TEXT_COL].notna()
].copy()

df_model_comparison = df_model_comparison.reset_index(drop=True)

print("Canciones a analizar:", len(df_model_comparison))

Canciones a analizar: 1378


In [449]:
# EJECUTAR

for config in MODEL_CONFIGS:

    model_results = run_model_on_df(
        df=df_model_comparison,
        model_config=config,
        text_col=TEXT_COL,
        device=device,
        batch_size=8,
        save_all_scores=False
    )

    df_model_comparison = pd.concat(
        [
            df_model_comparison.reset_index(drop=True),
            model_results.reset_index(drop=True)
        ],
        axis=1
    )

    display(
        df_model_comparison[
            [
                "artist",
                "song",
                config["output_col"],
                config["score_col"],
                f"{config['model_key']}_n_chunks"
            ]
        ].head()
    )

Cargando modelo: cardiffnlp/twitter-roberta-base-sentiment-latest


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 38261.56it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Etiquetas del modelo:
['negative', 'neutral', 'positive']


Aplicando cardiff: 100%|██████████| 1378/1378 [00:58<00:00, 23.50it/s]


,artist,song,cardiff_sentiment,cardiff_sentiment_score,cardiff_n_chunks
0,Susan Boyle,You Have To Be There,negative,0.611403,1
1,Taylor Swift,Cruel Summer,neutral,0.434510,2
2,Shania Twain,Forever And For Always,positive,0.850984,1
3,Johnny Cash,I Heard The Bells On Christmas Day,neutral,0.550447,1
4,Johnny Cash,Southwind,neutral,0.531417,1


...

Cargando modelo: SamLowe/roberta-base-go_emotions


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6902.25it/s]


Etiquetas del modelo:
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


Aplicando goemotions: 100%|██████████| 1378/1378 [00:56<00:00, 24.49it/s]


,artist,song,goemotions_top3,goemotions_top3_scores,goemotions_n_chunks
0,Susan Boyle,You Have To Be There,"confusion, curiosity, sadness","0.4627, 0.221, 0.2147",1
1,Taylor Swift,Cruel Summer,"sadness, love, annoyance","0.3982, 0.183, 0.1457",2
2,Shania Twain,Forever And For Always,"love, desire, caring","0.7028, 0.2402, 0.2331",1
3,Johnny Cash,I Heard The Bells On Christmas Day,"disappointment, sadness, realization","0.4433, 0.2922, 0.0573",1
4,Johnny Cash,Southwind,"excitement, joy, amusement","0.0542, 0.0256, 0.0177",1


In [450]:
go_top3_exploded = (
    df_model_comparison["goemotions_top3"]
    .dropna()
    .str.split(", ")
    .explode()
)

goemotions_top3_distribution = (
    go_top3_exploded
    .value_counts()
    .reset_index()
)

goemotions_top3_distribution.columns = ["goemotion", "n_appearances_top3"]

goemotions_top3_distribution["percentage"] = (
    goemotions_top3_distribution["n_appearances_top3"] 
    / goemotions_top3_distribution["n_appearances_top3"].sum() * 100
).round(2)

display(goemotions_top3_distribution)

,goemotion,n_appearances_top3,percentage
0,love,502,12.14
1,approval,493,11.93
2,sadness,431,10.43
3,disappointment,431,10.43
4,annoyance,351,8.49
5,desire,289,6.99
6,realization,192,4.64
7,curiosity,188,4.55
8,joy,175,4.23
9,caring,173,4.18


In [451]:
jhartmann_distribution = (
    df_model_comparison["jhartmann_emotion"]
    .value_counts()
    .reset_index()
)

jhartmann_distribution.columns = ["jhartmann_emotion", "n_songs"]

jhartmann_distribution["percentage"] = (
    jhartmann_distribution["n_songs"] / jhartmann_distribution["n_songs"].sum() * 100
).round(2)

display(jhartmann_distribution)

,jhartmann_emotion,n_songs,percentage
0,sadness,406,29.46
1,neutral,296,21.48
2,fear,274,19.88
3,anger,142,10.30
4,joy,139,10.09
5,surprise,97,7.04
6,disgust,24,1.74


In [452]:
cardiff_distribution = (
    df_model_comparison["cardiff_sentiment"]
    .value_counts()
    .reset_index()
)

cardiff_distribution.columns = ["cardiff_sentiment", "n_songs"]

cardiff_distribution["percentage"] = (
    cardiff_distribution["n_songs"] / cardiff_distribution["n_songs"].sum() * 100
).round(2)

display(cardiff_distribution)

,cardiff_sentiment,n_songs,percentage
0,neutral,526,38.17
1,positive,458,33.24
2,negative,394,28.59


In [454]:
display(
    df_model_comparison[
        [
            "artist",
            "song",
            "genre_list",
            "cardiff_sentiment",
            
            "jhartmann_emotion",
            
            "goemotions_top3",
            
        ]
    ].tail(20)
)

,artist,song,genre_list,cardiff_sentiment,jhartmann_emotion,goemotions_top3
1358,Jonny Lang,When I Come To You,"[blues rock, rock, blues]",neutral,sadness,"joy, love, approval"
1359,Moby,Jam For The Ladies,"[electronic, ambient, chillout]",positive,neutral,"approval, admiration, joy"
1360,Moby,The Sky Is Broken,"[electronic, ambient, chillout]",negative,fear,"sadness, love, disappointment"
1361,Kali Uchis,Melting,"[pop soul, dream pop, neo-soul]",positive,fear,"optimism, desire, curiosity"
1362,Air,Universal Traveler,"[electronic, chillout, ambient]",positive,joy,"admiration, joy, love"
1363,AC/DC,Thunderstruck,[hard rock],neutral,fear,"joy, amusement, excitement"
1364,AC/DC,Carry Me Home,[hard rock],neutral,fear,"amusement, annoyance, confusion"
1365,AC/DC,Safe In New York City,[hard rock],positive,joy,"approval, caring, joy"
1366,George Harrison,Don't Let Me Wait Too Long,"[classic rock, rock]",positive,sadness,"love, caring, desire"
1367,George Harrison,Let It Down,"[classic rock, rock]",positive,joy,"love, approval, admiration"


---

### 5. Agregación por género.

Después de calcular el sentimiento por canción, se agrupa por género.

Métricas recomendadas por género:

+ n_canciones

+ sentimiento_medio

+ sentimiento_mediano

+ desviación_típica

+ % Sentimiento_1

+ % Sentimiento_2

+ % Sentimiento_X

---

### 6. Visualización resultados

+ Barras: sentimiento medio por género

+ Boxplot - Violinplot por género

+ Barras apiladas % sentimientos

+ Nº canciones por género

# COSAS A REVISAR ANTES DE ACABAR

- Habría que revisar el preprocesado NLP de las lyrics, creo que no está mal pero quizás se puede afinar un poco.
- Decidir qué modelo usar para el análisis de sentimiento: GoEmotions es mejor sin duda además de que cuenta con 28 emociones distintas, en contra posición de las 7 del modelo más simple, no se si dejar ambos y hacer una comparación o simplemente quedarme con GoEmotions, quizas no hace falta compararlos ya que este es sólo un apartado de otros más del proyecto, pero creo que sería justo añadir una mención o insights breves sobre el modelo simple y su comparación con el modelo complejo.
- Arreglar la libreta y dejarla bonita que ahora mismo no está muy organizada.